Agentic AI Assignment

Daanish Khurshid

Atomcamp

28th June 2026

Disclaimer: For this assignment i moved away from the intial brief of a Senior Ai engineer building an enterprise grade tech support hub, So rather than using tech and software terminology, i used an industry where i have a little understanding, but the principles of what is trying to be acheived are implemented.

This project is based on Retrieval Augumented Generation (RAG) , and is being used to analyze complex financial statements and industrial annual reports of the 7 leading Pakistani enterprise sectors (Hubco, Engro, Sazgar, OGDC, Fauji Fertilizer, Lucky Cement and Systems). As i wanted to use a real world scenario that i could understand i choose this sector, with the advantage of finding real published company annual reports and financials. The idea is to make this as real as possible, and to see the performance of such a system. 

The system that has been developed is to first search the local database and then it falls back on the cloud based LLM. 

1. System Configuration and Setup

    Huggingface Embedding model "all-MiniLM-L6-v2" is being used for this project, as this is a compact sentence transformer , which can easily and quickly convert the sentences and paragraphs to 384 dimensional vector embeddings, which will much more efficient for our semenatic search engine. 

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load environment variables from the .env file
load_dotenv()

# Verify the Gemini API key is present in memory
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    print("Error: GEMINI_API_KEY not found in your .env file.")
else:
    print("Success: Gemini environment variables loaded.")

    try:
        # Initialize the free local embedding engine
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        print("Success: Local HuggingFace embedding engine initialized.")
        
        # Keep Gemini active strictly for the main chatbot logic
        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
        print("Success: Chat core reasoning engine initialized.")
    except Exception as e:
        print(f"Initialization Failed: {e}")


/Users/daanishkhurshid/miniconda3/envs/pak_industrial_rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/0q/hjy_4h2j4jzf1k_c5w04d4qc0000gn/T/ipykernel_72653/2107944698.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/var/folders/0q/hjy_4h2j4jzf1k_c5w04d4qc0000gn/T/ipykernel_72653/2107944698.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use i

Success: Gemini environment variables loaded.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4882.44it/s]


Success: Local HuggingFace embedding engine initialized.
Success: Chat core reasoning engine initialized.


2. Folder detection & Document Ingestion

The code will detect the folders and the documents that are downloaded into the folders. For this project we have on average 5 files for each company, two annual reports that are from the year 2024 and 2025 and the last 3 financial statements for the last nine months for each company. So the data is balanced for all companies.

The Logic behind this was to have just enough information to make this a real enterprise project having the lastest data, and also when you search for older data or some thing beyond 2026 the system will revert to calling the LLM after going through its own data base. 

Full page scans have been used for this project rather than using three or four sentences for the semantic database, shorter sentences were used but were causing issues with the api key.

In [3]:
import os
import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

# Set the target folder to the current working root directory
data_directory = "."
system_directories = {"venv", "chroma_db", ".git", "__pycache__", ".streamlit", "native_env"}

all_page_documents = []

# Scan the root path and filter out only company data folders
subfolders = [
    f.path for f in os.scandir(data_directory) 
    if f.is_dir() and os.path.basename(f.path) not in system_directories
]

print(f"Starting page-level ingestion. Scanning {len(subfolders)} folders...")

for folder_path in subfolders:
    raw_folder_name = os.path.basename(folder_path)
    clean_company_name = raw_folder_name.split(".")[-1].strip().lower().replace(" ", "_")
    
    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    
    for pdf_path in pdf_files:
        file_name = os.path.basename(pdf_path)
        try:
            # Load the PDF file page-by-page natively
            loader = PyPDFLoader(pdf_path)
            pages = loader.load()
            
            # Inject tracking data directly into the full page object
            for page in pages:
                page.metadata["company"] = clean_company_name
                page.metadata["source_file"] = file_name
                
            all_page_documents.extend(pages)
            print(f" Loaded {len(pages)} complete pages from File: {file_name}")
        except Exception as e:
            print(f"Skipping file {file_name} due to error: {e}")

print(f"\nSuccessfully compiled {len(all_page_documents)} whole-page data blocks.")

# Overwrite and initialize ChromaDB locally with the complete pages index
print("Generating vector metrics and writing clean index to disk...")
vector_store = Chroma.from_documents(
    documents=all_page_documents,
    embedding=embeddings, # Uses your fast, local HuggingFace embedding engine
    persist_directory="chroma_db"
)
print("🚀 SUCCESS: Page-Level ChromaDB successfully built and saved!")


Starting page-level ingestion. Scanning 7 folders...
 Loaded 46 complete pages from File: Transmission-of-Quarterly-Report-for-the-Period-Ended-December-31-2025.pdf
 Loaded 41 complete pages from File: Quarterly-Account-September-30-2025.pdf
 Loaded 413 complete pages from File: HUBCO-Integrated-Annual-Report-2025.pdf
 Loaded 384 complete pages from File: HUBCO-AR-Final-Low-Res.pdf
 Loaded 48 complete pages from File: Third-Quarter-Accounts-March-31-2026-–-The-Hub-Power-Company-Limited.pdf
 Loaded 55 complete pages from File: EHL-Final-Q1-2025-Report.pdf
 Loaded 185 complete pages from File: engro-holding-annual-report-2024.pdf
 Loaded 66 complete pages from File: EH-Q2-2025-report.pdf
 Loaded 178 complete pages from File: EH-Annual-Report-2025.pdf
 Loaded 62 complete pages from File: EH-Q3-2025-report.pdf
 Loaded 234 complete pages from File: Audited Annual Financial Statements for the year ended June 30, 2024.pdf
 Loaded 26 complete pages from File: (Un-Audited ) Quarterly & Nine Mon

Following code created 18000 data blocks, this was causing issues so, its commented out, will look into it later

In [ ]:
# # import os
# # import glob
# # from langchain_community.document_loaders import PyPDFLoader
# # from langchain_text_splitters import RecursiveCharacterTextSplitter
# # from langchain_community.vectorstores import Chroma

# # Set the target folder to the current working root directory
# data_directory = "."

# # Folders to completely skip during our company directory scan
# system_directories = {"venv", "chroma_db", ".git", "__pycache__", ".streamlit"}

# # Setup the text splitter configuration
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200,
#     length_function=len
# )

# all_processed_chunks = []

# # Scan the root path and filter out only company data folders
# subfolders = [
#     f.path for f in os.scandir(data_directory) 
#     if f.is_dir() and os.path.basename(f.path) not in system_directories
# ]

# print(f"Starting ingestion process. Found {len(subfolders)} candidate data folders to parse...")

# # Loop through each individual company folder
# for folder_path in subfolders:
#     raw_folder_name = os.path.basename(folder_path)
    
#     # Clean the folder names (e.g., "1. Hubco" -> "hubco")
#     clean_company_name = raw_folder_name.split(".")[-1].strip().lower().replace(" ", "_")
    
#     # Find all PDF files contained within this specific folder
#     pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
#     if not pdf_files:
#         continue
        
#     print(f"Processing company: '{clean_company_name}' | Found {len(pdf_files)} PDF reports.")
    
#     for pdf_path in pdf_files:
#         file_name = os.path.basename(pdf_path)
#         try:
#             # Load the text from the current PDF file
#             loader = PyPDFLoader(pdf_path)
#             loaded_documents = loader.load()
            
#             # Slice the document text into chunks
#             chunks = text_splitter.split_documents(loaded_documents)
            
#             # Inject structural tracking data to each individual chunk object
#             for chunk in chunks:
#                 chunk.metadata["company"] = clean_company_name
#                 chunk.metadata["source_file"] = file_name
                
#             all_processed_chunks.extend(chunks)
#         except Exception as e:
#             print(f"Skipping file {file_name} due to an error: {e}")

# print(f"Parsing complete. Successfully compiled {len(all_processed_chunks)} total text chunks.")

# # Initialize ChromaDB locally using the local memory embeddings
# print("Generating native embeddings and initializing local ChromaDB directory...")
# try:
#     vector_store = Chroma.from_documents(
#         documents=all_processed_chunks,
#         embedding=embeddings,
#         persist_directory="chroma_db"
#     )
#     print("Success: Vector store successfully built and saved locally inside 'chroma_db/'.")
# except Exception as e:
#     print(f"Database Initialization Failed: {e}")


Starting ingestion process. Found 7 candidate data folders to parse...
Processing company: 'hubco' | Found 5 PDF reports.
Processing company: 'engro' | Found 5 PDF reports.
Processing company: 'sazgar' | Found 6 PDF reports.
Processing company: 'ogdc' | Found 5 PDF reports.
Processing company: 'fauji' | Found 5 PDF reports.
Processing company: 'lucky' | Found 5 PDF reports.
Processing company: 'systems' | Found 5 PDF reports.
Parsing complete. Successfully compiled 18444 total text chunks.
Generating native embeddings and initializing local ChromaDB directory...
Success: Vector store successfully built and saved locally inside 'chroma_db/'.


3. Intializes Shared Memory state for multi agent system

Following code initializes the shared memory state for the multi-agent system and connects to the persistent vector database for semantic search
 
Multi agent is GraphState
TypedDict is to pass data between agents in the graph, this serves as the shared memory state. 

For the Retrieval-Augmented Generation (RAG) component, the system interfaces with a persistent Chroma vector database.

In [ ]:
from typing_extensions import TypedDict
from typing import List
from langchain_community.vectorstores import Chroma

# Define the shared memory space for your multi-agent system
class GraphState(TypedDict):
    original_query: str     # The original user query
    optimized_query: str    # The optimized query
    company_filter: str     # Search within a specific company
    documents: List[str]    # The text chunks from the vector database
    generation: str         # Final answer generated by the LLM
    loop_count: int         # Loop counter to prevent infinite loops

# Re-connect to your compiled database folder
print("Connecting to local vector database...")
db_connection = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)
print("Database connected. Vector search utilities are active.")


Connecting to local vector database...
Database connected. Vector search utilities are active.


4. Query Optimization and intent parsing

This is the Query optimization , this code translates ambiguous user inputs in to a structured form for optimized vector database retrieval. 

Model is assigned domain specific persona to ground its contextual understanding within the Pakistan Stock exchange(PSX)

The prompt is also used to keep within the selected companies and limit hallucinations

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Use double curly braces around the static JSON schema structure to escape them
rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a Senior AI Support Assistant specializing in the Pakistan Stock Exchange industrial sectors.\n"
     "Your task is to analyze an ambiguous, poorly phrased user technical query and optimize it.\n\n"
     "Perform two actions:\n"
     "1. Identify the target company name from the query. Map it strictly to one of these keys: "
     "['hubco', 'engro', 'sazgar', 'ogdc', 'fauji', 'lucky', 'systems', 'unknown'].\n"
     "2. Rewrite the messy query into highly descriptive, keyword-driven search variations optimized for vector database distance matching.\n\n"
     "You MUST respond with a valid JSON object matching this schema exactly:\n"
     "{{\n"
     "  \"company\": \"company_key\",\n"
     "  \"optimized_query\": \"descriptive keyword query string\"\n"
     "}}\n"
     "Do not include any markdown styling, wrappers, or text outside the raw JSON structure."),
    ("human", "Messy Customer Query: {query}")
])

# Create the executable pipeline chain By piping the raw LLM output directly into a 
# JsonOutputParser, the system guarantees that the unstructured text generation from 
# the language model is programmatically cast into a native Python dictionary at runtime. 
# This mitigates formatting errors common in raw string extraction.

rewriter_chain = rewriter_prompt | llm | JsonOutputParser()

# Define the formal LangGraph node function
def rewrite_query_node(state: GraphState) -> dict:
    """
    Reads the raw query from state, optimizes it using the LLM, 
    and updates the state object with the search keywords.
    """
    print("\n--- NODE: REWRITE QUERY (Support Agent) ---")
    current_raw_input = state["original_query"]
    print(f"Original Input: '{current_raw_input}'")
    
    # Execute the LLM rewriting pipeline
    llm_output = rewriter_chain.invoke({"query": current_raw_input})
    
    print(f"Identified Target Company: {llm_output['company']}")
    print(f"Optimized Search Keywords: {llm_output['optimized_query']}")
    
    # Return the updated state fields
    return {
        "optimized_query": llm_output["optimized_query"],
        "company_filter": llm_output["company"],
        "loop_count": state.get("loop_count", 0)
    }


5. Testing and Validation of Node using a mock initial state

In [8]:
mock_initial_state = {
    "original_query": "sazgar ev car selling metrics and plant issues",
    "loop_count": 0
}

updated_state_output = rewrite_query_node(mock_initial_state)
print("\nReturned Node State Dictionary updates:")
print(updated_state_output)



--- NODE: REWRITE QUERY (Support Agent) ---
Original Input: 'sazgar ev car selling metrics and plant issues'
Identified Target Company: sazgar
Optimized Search Keywords: Sazgar Engineering Works electric vehicle sales performance metrics, EV unit sales volume, market share, revenue from EV segment, sales growth trends, EV sales data, manufacturing plant operational challenges, production disruptions, supply chain issues, factory output problems, plant efficiency, production capacity constraints, operational bottlenecks, manufacturing facility issues

Returned Node State Dictionary updates:
{'optimized_query': 'Sazgar Engineering Works electric vehicle sales performance metrics, EV unit sales volume, market share, revenue from EV segment, sales growth trends, EV sales data, manufacturing plant operational challenges, production disruptions, supply chain issues, factory output problems, plant efficiency, production capacity constraints, operational bottlenecks, manufacturing facility is

6. Defensive Guardrails and Prompting Strategy

This configuration serves two critical functions in production-grade multi-agent frameworks, 

Computational Efficiency: Passing full blocks of irrelevant text into a final generation prompt wastes a massive amount of token bandwidth and increases processing costs. By routing chunks through a lightweight, binary check first, the system filters out noise early.

Adaptive Graph Control: In a complete LangGraph implementation, the output of this grader_chain provides the conditional routing logic. If the relevance_score is consistently "no", the state graph can dynamically detour the workflow back to the query optimization node for a retry, rather than outputting a broken answer to the user.🎓

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Instruct Gemini to act as a strict QA Document Evaluator
grader_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert Quality Assurance Engineer grading the relevance of a retrieved document chunk.\n"
     "Analyze if the provided document text contains facts, metrics, or context useful for answering the user question.\n\n"
     "Provide a strict binary score evaluating document relevance.\n"
     "You MUST respond with a valid JSON object matching this schema exactly:\n"
     "{{\n"
     "  \"relevance_score\": \"yes\" | \"no\"\n"
     "}}\n"
     "Output ONLY the raw JSON string. Do not include conversational text, markdown symbols, or code fences."),
    ("human", "Retrieved Document Text:\n\n{document}\n\nUser Question: {question}")
])

# Create the executable grading pipeline chain
grader_chain = grader_prompt | llm | JsonOutputParser()
print("QA Document Grader Pipeline compiled and ready.")


QA Document Grader Pipeline compiled and ready.


7. Retrieval and Filtering Node 

The retrieve_and_grade_node function represents a composite operational node within the multi-agent system. It unifies isolated database infrastructure queries with the active automated quality evaluation layer to construct a refined contex.

Upon completion, the node returns a state update containing only the verified context pool ({"documents": relevant_documents}). This pattern isolates the framework state updates, ensuring that subsequent generation nodes only have access to facts that have been explicitly verified by the QA layer.

In [11]:
def retrieve_and_grade_node(state: GraphState) -> dict:
    """
    Support Agent retrieves top 3 documents from ChromaDB.
    QA Agent evaluates each chunk and filters out irrelevant context.
    """
    print("\n--- NODE: RETRIEVE & GRADE (Collaboration Network) ---")
    
    query_keywords = state["optimized_query"]
    target_company = state["company_filter"]
    original_user_input = state["original_query"]
    
    print(f"Executing Vector Search for: '{target_company}' documents...")
    
    # Configure a secure metadata filter so we only search the target company's documents
    search_filter = {} if target_company == "unknown" else {"company": target_company}
    
    # Pull the top 3 closest matching textual chunks
    retrieved_chunks = db_connection.similarity_search(
        query=query_keywords,
        k=3,
        filter=search_filter
    )
    
    relevant_documents = []
    
    # QA Agent runs an explicit assessment pass on every chunk
    for idx, doc in enumerate(retrieved_chunks):
        chunk_text = doc.page_content
        source_file = doc.metadata.get("source_file", "unknown")
        
        # Invoke the QA evaluation chain
        grade_result = grader_chain.invoke({"document": chunk_text, "question": original_user_input})
        score = grade_result.get("relevance_score", "no").strip().lower()
        
        print(f" -> Chunk [{idx+1}] from File '{source_file}' | Grade Score: {score.upper()}")
        
        if score == "yes":
            relevant_documents.append(chunk_text)
            
    print(f"Evaluation Complete. Kept {len(relevant_documents)} out of 3 retrieved chunks.")
    
    # Update the state memory payload
    return {"documents": relevant_documents}


8. Pipeline Integration Testing

To verify that the isolated agents cooperate seamlessly, the system undergoes an integration test. This test feeds the actual output of the Query Optimization Node directly into the input space of the Retrieval & Grading Node, simulating a real runtime state transition.

In [12]:
# Pass the state updates generated by your query rewriter node test
mock_state_after_rewriting = {
    "original_query": "sazgar ev car selling metrics and plant issues",
    "optimized_query": updated_state_output["optimized_query"],
    "company_filter": updated_state_output["company_filter"]
}

# Execute the Retrieval & Grading node manually
graded_state_output = retrieve_and_grade_node(mock_state_after_rewriting)

print("\nReturned Node State Dictionary updates:")
print(f"Total documents kept in state: {len(graded_state_output['documents'])}")



--- NODE: RETRIEVE & GRADE (Collaboration Network) ---
Executing Vector Search for: 'sazgar' documents...
 -> Chunk [1] from File 'Audited Annual Financial Statements for the year ended June 30, 2025 WEB.pdf' | Grade Score: NO
 -> Chunk [2] from File 'Audited Annual Financial Statements for the year ended June 30, 2024.pdf' | Grade Score: NO
 -> Chunk [3] from File 'Audited Annual Financial Statements for the year ended June 30, 2024.pdf' | Grade Score: NO
Evaluation Complete. Kept 0 out of 3 retrieved chunks.

Returned Node State Dictionary updates:
Total documents kept in state: 0


9. Fallback Architecture via Real-Time Web Grounding

To guarantee system resilience when local vector repositories lack sufficient or up-to-date documentation, the pipeline incorporates an automated Web Search Fallback Node. This component transforms a standard closed-world Retrieval-Augmented Generation (RAG) pipeline into a flexible, open-world hybrid retrieval system.

Temperature is set to zero , to suppress creative text styling and to force the model to synthesize search results strictly based on extracted web citations.

Production-grade multi-agent architectures must account for API rate limits, network timeouts, or credential drops. The node wraps the entire pipeline invocation inside a defensive try-except block. If the Google GenAI connection fails, the system logs the error telemetry (Gemini Native Web Search Failed) and gracefully returns the unmodified state (current_documents). This structural boundary prevents an external network error from crashing the entire LangGraph execution pipeline

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Configure Gemini to use native Google Search grounding via model_kwargs
gemini_search_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.0,
    model_kwargs={"google_search_enabled": True}  # Safely injects the background flag
)

def web_search_node(state: GraphState) -> dict:
    """
    If local documentation is insufficient, the Support Agent triggers a live 
    Google Search fallback using Gemini's native grounding tools.
    """
    print("\n--- NODE: WEB SEARCH FALLBACK (Support Agent via Google Gemini) ---")
    
    search_keywords = state["optimized_query"]
    current_documents = state.get("documents", [])
    
    print(f"Triggering native Google Search grounding for: '{search_keywords}'")
    
    try:
        # Prompt Gemini to perform a search and summarize the findings
        search_prompt = (
            f"Perform a live web search regarding this topic and extract the latest "
            f"technical/operational/financial updates: {search_keywords}.\n"
            f"Provide a clear factual summary of your findings."
        )
        
        response = gemini_search_llm.invoke(search_prompt)
        web_summary = response.content
        
        # Append the fresh live context into our existing state document array
        updated_documents = list(current_documents)
        updated_documents.append(web_summary)
        
        print("Live Google Search extraction complete. Appended real-time context to state.")
        return {"documents": updated_documents}
        
    except Exception as e:
        print(f"Gemini Native Web Search Failed: {e}")
        return {"documents": current_documents}


10. Unit integration testing
    
    This following code isnt necessary, its for the testing the core architectual concept

In [ ]:
# Define the routing logic function
def route_after_grading(state: dict) -> str:
    """
    Evaluates the document count. If the QA Agent approved local context, 
    route to generate. If state documents are empty, trigger the live web fallback.
    """
    print("\n--- CONDITIONAL EDGE: EVALUATING ROUTE PATH ---")
    available_docs = state.get("documents", [])
    
    if not available_docs:
        print("Path Chosen: [Path B] -> Local Documents Insufficient. Routing to Web Search.")
        return "web_search"
    else:
        print(f"Path Chosen: [Path A] -> Found {len(available_docs)} Grounded Chunk(s). Routing to Generator.")
        return "generate"

# 2. Simulate a state layout where no local documents passed the QA test
mock_failed_grading_state = {
    "optimized_query": "Sazgar Engineering electric vehicle assembly plant updates 2026",
    "documents": []
}

# 3. Test your routing logic condition function
next_step = route_after_grading(mock_failed_grading_state)

# 4. Execute the Gemini native web node if the search track is chosen
if next_step == "web_search":
    web_state_output = web_search_node(mock_failed_grading_state)
    print(f"\nReturned Node State updates: Successfully appended live summary context.")
    print("\n--- Extract of Web Content Appended ---")
    print(web_state_output["documents"][-1][:500] + "...")



--- CONDITIONAL EDGE: EVALUATING ROUTE PATH ---
Path Chosen: [Path B] -> Local Documents Insufficient. Routing to Web Search.

--- NODE: WEB SEARCH FALLBACK (Support Agent via Google Gemini) ---
Triggering native Google Search grounding for: 'Sazgar Engineering electric vehicle assembly plant updates 2026'
Live Google Search extraction complete. Appended real-time context to state.

Returned Node State updates: Successfully appended live summary context.

--- Extract of Web Content Appended ---
Performing a live web search for "Sazgar Engineering electric vehicle assembly plant updates 2026" and related terms, here is a factual summary of the latest available information:

**Sazgar Engineering Electric Vehicle Assembly Plant Updates (Focusing on recent developments and future outlook towards 2026)**

Sazgar Engineering Works Limited, a prominent Pakistani automotive manufacturer, has been actively involved in the electric vehicle (EV) sector, primarily through its partnership with Chi

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Design a prompt forcing strict grounding against state documents
generator_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Senior Technical Support Specialist for Pakistan Stock Exchange industrial assets.\n"
     "Synthesize a clear, highly accurate, and professional technical answer to the user's question.\n"
     "You MUST base your response strictly on the provided context passages (local data and web summaries).\n"
     "If the context does not contain enough data to formulate an answer, state that you do not know.\n"
     "Do not invent or extrapolate any financial or operational data points.\n\n"
     "Context Passages:\n{context}"),
    ("human", "User Question: {question}")
])

generator_chain = generator_prompt | llm | StrOutputParser()

# 2. Define the formal LangGraph node function
def generate_answer_node(state: GraphState) -> dict:
    """
    Support Agent drafts the comprehensive technical answer 
    using the accumulated state documents as a strict reference.
    """
    print("\n--- NODE: GENERATOR (Support Agent) ---")
    
    question = state["original_query"]
    compiled_docs = state.get("documents", [])
    
    # Combine all text chunks into a single reference block for the LLM
    context_block = "\n\n---\n\n".join(compiled_docs)
    
    print(f"Drafting answer using {len(compiled_docs)} source context blocks...")
    generated_response = generator_chain.invoke({
        "context": context_block,
        "question": question
    })
    
    # Return the generated text to update the state memory
    return {"generation": generated_response}


In [16]:
# 1. Design the QA Gatekeeper evaluation prompt
hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an automated Quality Assurance Judge evaluating an AI-generated technical support answer.\n"
     "Your task is to verify if the generated response is perfectly grounded in and supported by the source context documents.\n\n"
     "Perform a strict alignment check:\n"
     "- If the generated answer contains any fact, metric, or statement NOT found in the source documents, it is a hallucination.\n"
     "- If the answer states it does not know because of insufficient context, that is NOT a hallucination (pass it).\n\n"
     "You MUST respond with a valid JSON object matching this schema exactly:\n"
     "{{\n"
     "  \"hallucination_check\": \"passed\" | \"failed\"\n"
     "}}\n"
     "Output ONLY the raw JSON string. Do not include conversational text or markdown wrappers."),
    ("human", "Source Context Documents:\n\n{context}\n\nGenerated Answer Draft:\n\n{generation}")
])

hallucination_chain = hallucination_prompt | llm | JsonOutputParser()

# 2. Define the formal LangGraph node function
def qa_hallucination_node(state: GraphState) -> dict:
    """
    QA Agent compares the draft generation against source documents to catch hallucinations.
    If it fails, it increments the self-correction loop counter.
    """
    print("\n--- NODE: QA HALLUCINATION CHECK (QA Gatekeeper) ---")
    
    compiled_docs = state.get("documents", [])
    draft_generation = state["generation"]
    current_loops = state.get("loop_count", 0)
    
    context_block = "\n\n---\n\n".join(compiled_docs)
    
    print("Reviewing answer grounding against sources...")
    judge_result = hallucination_chain.invoke({
        "context": context_block,
        "generation": draft_generation
    })
    
    status = judge_result.get("hallucination_check", "failed").strip().lower()
    print(f"QA Evaluation Result: {status.upper()}")
    
    # If it fails, we update the state with an incremented loop count
    if status == "failed":
        new_loop_count = current_loops + 1
        print(f"⚠️ Hallucination detected! Incrementing loop counter to: {new_loop_count}")
        return {"generation": status, "loop_count": new_loop_count}
        
    return {"generation": status, "loop_count": current_loops}


11. End to End System Generation and Hallucination verification

To validate the final stages of the multi-agent pipeline, the system undergoes an integrated execution trace. This test routes the aggregated context payloads gathered by previous nodes directly into the generation engine, immediately followed by an automated Hallucination Guardrail Check.

In [17]:
# 1. Simulate passing the web search results into the generator node input
mock_generator_state = {
    "original_query": "sazgar ev car selling metrics and plant issues",
    "documents": web_state_output["documents"],
    "loop_count": 0
}

# 2. Execute the Generator Node
generation_output = generate_answer_node(mock_generator_state)
print(f"\nDraft Generation Output:\n{generation_output['generation']}")

# 3. Simulate passing that draft into the QA Hallucination Check Node
mock_qa_state = {
    "documents": web_state_output["documents"],
    "generation": generation_output["generation"],
    "loop_count": 0
}

qa_output = qa_hallucination_node(mock_qa_state)
print(f"\nFinal State Variable Updates: {qa_output}")



--- NODE: GENERATOR (Support Agent) ---
Drafting answer using 1 source context blocks...

Draft Generation Output:
Based on the provided context, here is the information regarding Sazgar's EV car selling metrics and plant issues:

**Sazgar EV Car Selling Metrics:**
*   The BAIC EX5 EV is a relatively new entrant to the Pakistani market, having been officially launched in late 2023/early 2024.
*   Due to its recent introduction, detailed sales figures specifically for the EV segment are still emerging and are not publicly detailed in the provided information.
*   The company's financial reports indicate ongoing investments and operational costs associated with its automotive ventures, which include the EV segment.

**Sazgar EV Plant Issues:**
*   The provided context does not mention any specific "plant issues" or operational problems related to Sazgar's electric vehicle assembly facilities.
*   Sazgar utilizes its existing automotive assembly plant in Lahore for the assembly of BAIC v

In [18]:
def route_after_hallucination_check(state: GraphState) -> str:
    """
    Reads the generation assessment flag and loop_count state variables 
    to determine if the system should exit or trigger a self-correction loop.
    """
    print("\n--- CONDITIONAL EDGE: EVALUATING GROUNDING ACCURACY ---")
    
    # In our upcoming graph, the node puts the status ('passed'/'failed') in state["generation"]
    check_status = state["generation"].strip().lower()
    retry_attempts = state.get("loop_count", 0)
    
    if check_status == "passed":
        print("Path Chosen: Grounding Verified. Routing to [END].")
        return "perfect_exit"
        
    print(f"Path Chosen: Grounding Failed (Self-Correction Loop). Current Loop Count: {retry_attempts}")
    if retry_attempts >= 2:
        print("⚠️ Maximum self-correction retry limit (2 attempts) reached! Forcing graceful fallback exit.")
        return "forced_exit"
    else:
        print("🔄 Rerouting back to Node 1 (Query Rewriter) for an automated rewrite retry.")
        return "retry_loop"


12. Framework Compilation and Global Graph Topology Synchronization

In [19]:
from langgraph.graph import StateGraph, END

# 1. Update the routing logic function to return 'generate_answer' to match the graph configuration keys
def route_after_grading(state: GraphState) -> str:
    """
    Evaluates the document count. If the QA Agent approved local context, 
    route to generate_answer. If state documents are empty, trigger the live web fallback.
    """
    print("\n--- CONDITIONAL EDGE: EVALUATING ROUTE PATH ---")
    available_docs = state.get("documents", [])
    
    if not available_docs:
        print("Path Chosen: [Path B] -> Local Documents Insufficient. Routing to Web Search.")
        return "web_search"
    else:
        print(f"Path Chosen: [Path A] -> Found {len(available_docs)} Grounded Chunk(s). Routing to Generator.")
        return "generate_answer"  # Fixed string return target name

# 2. Initialize the StateGraph structure
workflow = StateGraph(GraphState)

# 3. Register all operational agent nodes into the graph workspace
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("retrieve_and_grade", retrieve_and_grade_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate_answer", generate_answer_node)
workflow.add_node("qa_hallucination", qa_hallucination_node)

# 4. Establish the starting point entry line
workflow.set_entry_point("rewrite_query")

# 5. Connect the deterministic straight-line engineering edges
workflow.add_edge("rewrite_query", "retrieve_and_grade")
workflow.add_edge("web_search", "generate_answer")

# 6. Connect Conditional Edge 1: Routes after document grading
workflow.add_conditional_edges(
    "retrieve_and_grade",
    route_after_grading,
    {
        "web_search": "web_search",
        "generate_answer": "generate_answer"
    }
)

# 7. Connect Conditional Edge 2: Routes after QA hallucination inspection
workflow.add_conditional_edges(
    "qa_hallucination",
    route_after_hallucination_check,
    {
        "perfect_exit": END,
        "forced_exit": END,      
        "retry_loop": "rewrite_query"
    }
)

# 8. Add a bridge link to feed the generator output straight into the QA judge node
workflow.add_edge("generate_answer", "qa_hallucination")

# 9. Compile the graph topology mesh into a ready-to-run execution engine
app_graph = workflow.compile()
print("🚀 SUCCESS: LangGraph Multi-Agent Self-Correction Network Compiled and Synchronized Perfectly!")


🚀 SUCCESS: LangGraph Multi-Agent Self-Correction Network Compiled and Synchronized Perfectly!


In [20]:
initial_input = {
    "original_query": "sazgar ev car selling metrics and plant issues",
    "loop_count": 0
}

print("Initiating full multi-agent network streaming trace...\n")
final_output_state = app_graph.invoke(initial_input)
print("\n=================== GRAPH RUN COMPLETE ===================")


Initiating full multi-agent network streaming trace...


--- NODE: REWRITE QUERY (Support Agent) ---
Original Input: 'sazgar ev car selling metrics and plant issues'
Identified Target Company: sazgar
Optimized Search Keywords: Sazgar Engineering Works electric vehicle sales performance metrics, EV unit sales volume, market share, revenue from EV segment, manufacturing plant operational issues, production bottlenecks, supply chain disruptions, factory output problems, facility maintenance challenges, production capacity constraints

--- NODE: RETRIEVE & GRADE (Collaboration Network) ---
Executing Vector Search for: 'sazgar' documents...
 -> Chunk [1] from File 'Audited Annual Financial Statements for the year ended June 30, 2025 WEB.pdf' | Grade Score: YES
 -> Chunk [2] from File 'Audited Annual Financial Statements for the year ended June 30, 2024.pdf' | Grade Score: NO
 -> Chunk [3] from File 'Audited Annual Financial Statements for the year ended June 30, 2025 WEB.pdf' | Grade Score:

ScreenShots of the Steamlit app

Each company can be selected from the side drop down menu